# 1. Pré-processamento de dados

De sequência de DNA, que é texto, para tensor, que é número.

Ao final desta seção teremos um array de forma `(55001, 201, 4)`, com cada
número carregando um significado biológico preciso.

## 1.1. Dados do genoma do cupuaçu

O cupuaçu (*Theobroma grandiflorum*) é uma árvore frutífera amazônica, parente
próxima do cacau.

<img src="https://blog.iprocess.com.br/wp-content/uploads/2021/11/placeholder.png" width="620">

<!-- TROCAR: Figura 1 de Alves et al. 2024 - (A) árvore, (B) fruto, (C) fruto aberto -->

Genoma montado telômero-a-telômero, publicado em Alves et al., *GigaScience*
13:giae027, 2024. As janelas de sequência usadas aqui saíram dele por
`scripts/extract.py`, que roda fora da aula porque precisa ler os 424 Mb
inteiros.

In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt

# No Colab: faça o upload de splice_donor.npz ou monte o Google Drive.
CAMINHO_DADOS = "data/splice_donor.npz"

dados = np.load(CAMINHO_DADOS)

# O .npz carrega junto um dicionário de metadados, gravado pelo script de
# extração. É de lá que vêm todos os números citados nesta seção: nenhum
# deles está digitado à mão no notebook.
meta = json.loads(str(dados["meta"]))
genoma = meta["genome_stats"]


def br(numero):
    """Formata inteiro no padrão brasileiro: 1234567 -> 1.234.567"""
    return f"{numero:,}".replace(",", ".")


print(f"Pares de base:      {br(genoma['genome_bp'])}")
print(f"Cromossomos:        {genoma['n_chromosomes']}")
print(f"Genes anotados:     {br(genoma['annotated_genes'])}")
print(f"Íntrons anotados:   {br(genoma['annotated_introns'])}")

## 1.2. Problema de encontrar sítios doadores em genomas

Um gene não vira proteína diretamente:

1. o gene é transcrito em RNA;
2. esse RNA tem **éxons**, que permanecem, e **íntrons**, que são removidos;
3. cortar os íntrons e emendar os éxons é o **splicing**.

```
DNA / pré-mRNA:   [éxon 1][íntron 1][éxon 2][íntron 2][éxon 3]
                          \________/        \________/
                           removido           removido

mRNA maduro:      [éxon 1][éxon 2][éxon 3]
```

O **sítio doador** é a fronteira onde um íntron começa, e é ele que vamos
prever.

Quase todo íntron começa com o dinucleotídeo `GT`: 98,57% dos 119.378 anotados
neste genoma. Parece que bastaria procurar `GT`.

In [ ]:
# Contagens sobre o genoma INTEIRO, calculadas pelo script de extração.
grupos = [
    ("Todos os GT\ndo genoma", genoma["gt_genome_wide"]),
    ("GT dentro\nde genes", genoma["gt_in_gene_bodies"]),
    ("Sítios doadores\nreais", genoma["donor_sites"]),
]
rotulos = [nome for nome, _ in grupos]
valores = [valor for _, valor in grupos]

# Rampa sequencial de um único tom: os três grupos são subconjuntos aninhados
# da mesma medida, do mais geral (claro) ao mais específico (escuro).
CORES = ["#86b6ef", "#2a78d6", "#104281"]
SUPERFICIE, TINTA, MUTED, GRADE = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"

figura, eixos = plt.subplots(1, 2, figsize=(12, 3.4), facecolor=SUPERFICIE)
posicao = range(len(grupos))

for indice, eixo in enumerate(eixos):
    eixo.barh(posicao, valores, color=CORES, height=0.62)
    eixo.set_yticks(posicao)
    eixo.set_yticklabels(rotulos, fontsize=9, color=TINTA)
    eixo.invert_yaxis()
    eixo.set_facecolor(SUPERFICIE)
    eixo.xaxis.grid(True, color=GRADE, linewidth=0.8)
    eixo.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        eixo.spines[lado].set_visible(False)
    eixo.spines["bottom"].set_color(GRADE)
    eixo.tick_params(colors=MUTED, labelsize=8)

eixos[0].set_title("Escala linear", fontsize=10, color=TINTA, loc="left")
eixos[0].set_xlabel("ocorrências", fontsize=8, color=MUTED)
eixos[0].set_xlim(0, max(valores) * 1.28)

eixos[1].set_xscale("log")
eixos[1].set_xlim(1e4, 1e8)
eixos[1].set_title("Escala logarítmica", fontsize=10, color=TINTA, loc="left")
eixos[1].set_xlabel("ocorrências (log)", fontsize=8, color=MUTED)

# Os dois painéis rotulam coisas diferentes, para não repetir informação:
# à esquerda a proporção, à direita a contagem absoluta.
total = max(valores)
for indice, valor in enumerate(valores):
    eixos[0].text(valor + total * 0.02, indice,
                  f"{valor / total * 100:.2f}%".replace(".", ","),
                  va="center", fontsize=9, color=TINTA)
    eixos[1].text(valor * 1.25, indice, br(valor), va="center",
                  fontsize=9, color=TINTA)

figura.suptitle("Quantos GT existem, e quantos são sítio de splice",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

razao_genoma = genoma["gt_genome_wide"] / genoma["donor_sites"]
razao_genes = genoma["gt_in_gene_bodies"] / genoma["donor_sites"]
print(f"No genoma inteiro:  1 sítio real a cada {razao_genoma:.0f} GT")
print(f"Dentro de genes:    1 sítio real a cada {razao_genes:.0f} GT")

Os mesmos três números, em duas escalas:

- linear: os sítios reais somem, porque são **0,29%** de todos os `GT`;
- logarítmica: ficam legíveis, mas o abismo entre as barras desaparece.

Guarde esse par de painéis. Na seção 2 o mesmo fenômeno reaparece, com duas
métricas no lugar de duas escalas.

> Achar `GT` é trivial. Decidir qual `GT` é sítio doador é o problema, e a
> única informação disponível para decidir é a sequência ao redor.

## 1.3. Janelas de sequências genômicas

Um modelo não recebe o genoma inteiro, e sim **janelas** de tamanho fixo. As
deste dataset têm 201 bases, ancoradas em um `GT`:

```
[ 100 bases do lado do éxon ][G T][ 99 bases entrando no íntron ]
                              ^
                        posição 100
```

O `GT` fica sempre na posição 100, tanto nos positivos quanto nos negativos.
Se os negativos fossem trechos aleatórios do genoma, bastaria ao modelo
procurar `GT`. Ancorados todos no mesmo lugar, ele precisa decidir pelo
contexto.

In [ ]:
X_treino = dados["X_train"]   # sequências, ainda como texto
y_treino = dados["y_train"]   # 1 = sítio doador real, 0 = não é

# As sequências estão guardadas como bytes; .decode() devolve texto.
primeira = X_treino[0].decode()

print("Uma janela crua:")
print(primeira)
print()
print(f"Comprimento:        {len(primeira)} bases")
print(f"Posição 100 e 101:  {primeira[100:102]}")
print(f"Rótulo:             {y_treino[0]}")

Seis janelas que são sítio doador e seis que não são, mostradas na região
central.

O que o primeiro grupo tem que o segundo não tem?

In [ ]:
POSICAO_DOADOR = meta["donor_offset"]   # 100


def mostrar_centro(sequencia, margem=10):
    """Recorta a região central da janela e separa o GT com colchetes."""
    esquerda = sequencia[POSICAO_DOADOR - margem:POSICAO_DOADOR]
    direita = sequencia[POSICAO_DOADOR + 2:POSICAO_DOADOR + 2 + margem]
    meio = sequencia[POSICAO_DOADOR:POSICAO_DOADOR + 2]
    return f"{esquerda}[{meio}]{direita}"


print("SÍTIOS DOADORES REAIS")
for janela in X_treino[y_treino == 1][:6]:
    print("   ", mostrar_centro(janela.decode()))

print("\nNÃO SÃO SÍTIO")
for janela in X_treino[y_treino == 0][:6]:
    print("   ", mostrar_centro(janela.decode()))

Você provavelmente viu `AG` logo antes do `GT` em várias do primeiro grupo, e
`AAG` logo depois em algumas. É o consenso do sítio doador, escrito na
literatura como `MAG|GTAAGT`.

Mas o padrão não está em todas. Melhor medir do que confiar na impressão.

In [ ]:
def frequencia(janelas, trecho, inicio):
    """Fração das janelas que contêm `trecho` a partir da posição `inicio`."""
    fim = inicio + len(trecho)
    return np.mean([j.decode()[inicio:fim] == trecho for j in janelas]) * 100


positivas = X_treino[y_treino == 1]
negativas = X_treino[y_treino == 0]

print(f"{'padrão':<22} {'nos sítios':>11} {'nos não-sítios':>15}")
print("-" * 50)
for descricao, trecho, inicio in [
    ("AG logo antes do GT", "AG", POSICAO_DOADOR - 2),
    ("GTA", "GTA", POSICAO_DOADOR),
    ("GTAAG", "GTAAG", POSICAO_DOADOR),
    ("GTAAGT (consenso)", "GTAAGT", POSICAO_DOADOR),
]:
    print(f"{descricao:<22} {frequencia(positivas, trecho, inicio):9.1f}% "
          f"{frequencia(negativas, trecho, inicio):14.1f}%")

O problema da aula, agora em números:

- exigir o consenso completo `GTAAGT` perderia 91 de cada 100 sítios reais;
- `AG` antes do `GT` acha metade dos sítios, mas também aparece em 5,9% dos
  não-sítios, que são 50 vezes mais numerosos.

Nenhuma regra fixa resolve. Todo padrão visível é uma tendência, não um
critério.

Por isso o consenso não é uma sequência, e sim uma distribuição de frequências.
É exatamente o que a seção 3 vai construir.

## 1.4. One-hot encoding

Redes neurais operam sobre números, não sobre letras.

A tradução ingênua `A=0, C=1, G=2, T=3` está errada: ela inventa uma ordem e
uma distância que não existem na biologia. Afirma que `T` é três vezes `C`, e
que `A` está mais longe de `T` do que de `C`. O modelo acreditaria.

No **one-hot** cada base vira um vetor com um único 1, o que deixa as quatro
equidistantes entre si:

| base | vetor |
|---|---|
| A | `[1, 0, 0, 0]` |
| C | `[0, 1, 0, 0]` |
| G | `[0, 0, 1, 0]` |
| T | `[0, 0, 0, 1]` |

Uma janela de 201 bases vira, então, uma matriz `(201, 4)`.

### 1.4.1. Lacuna: escrever a função de one-hot

Complete a função abaixo: ela recebe uma sequência de DNA como texto e devolve
a matriz `(comprimento, 4)`.

- `BASES.index("C")` devolve `1`, que é a coluna certa para a base `C`.
- `matriz[posicao, coluna] = 1.0` marca uma célula da matriz.

In [ ]:
BASES = "ACGT"


def one_hot(sequencia):
    matriz = np.zeros((len(sequencia), 4), dtype=np.float32)
    for posicao, base in enumerate(sequencia):
        # SEU CÓDIGO AQUI: marque com 1.0 a coluna correspondente a `base`.
        pass
    return matriz

In [ ]:
# Verificação. Se a função estiver certa, "ACGT" vira a matriz identidade.
resultado = one_hot("ACGT")
esperado = np.eye(4, dtype=np.float32)

print(resultado)
if np.array_equal(resultado, esperado):
    print("\nCorreto: cada base virou um vetor com um único 1.")
else:
    print("\nAinda não. Cada linha precisa ter exatamente um 1.0,")
    print("na coluna dada por BASES.index(base).")

O laço acima serve para entender. Para converter os 55 mil exemplos de uma vez
ele seria lento demais.

A versão vetorizada faz a mesma operação de uma vez só, com numpy.

In [ ]:
def one_hot_lote(sequencias):
    """Converte um array inteiro de sequências para one-hot, de uma vez.

    Devolve um array de forma (n_exemplos, comprimento, 4).
    """
    # Cada caractere vira seu código ASCII: 'A' -> 65, 'C' -> 67, ...
    codigos = sequencias.view(np.uint8).reshape(len(sequencias), -1)

    # Tabela que traduz código ASCII -> coluna do one-hot.
    tabela = np.zeros(256, dtype=np.uint8)
    for coluna, base in enumerate(BASES):
        tabela[ord(base)] = coluna

    return np.eye(4, dtype=np.float32)[tabela[codigos]]


X_treino_oh = one_hot_lote(X_treino)
print("Forma do conjunto de treino:", X_treino_oh.shape)

## 1.5. Formato dos dados

Os três eixos do array, que é a leitura que se repete no resto da aula:

In [ ]:
n_exemplos, comprimento, n_bases = X_treino_oh.shape

print(f"({n_exemplos}, {comprimento}, {n_bases})")
print()
print(f"  eixo 0 = {n_exemplos:>6} janelas do conjunto de treino")
print(f"  eixo 1 = {comprimento:>6} posições dentro de cada janela")
print(f"  eixo 2 = {n_bases:>6} bases possíveis (A, C, G, T)")

Checagem de sanidade: se cada posição tem exatamente uma base, somar ao longo
do último eixo tem que dar 1 em toda posição.

In [ ]:
somas = X_treino_oh.sum(axis=2)
print("Toda posição soma exatamente 1:", bool(np.all(somas == 1.0)))

# E a âncora: todo exemplo tem G na posição 100 e T na 101.
coluna_G, coluna_T = BASES.index("G"), BASES.index("T")
tem_G = np.all(X_treino_oh[:, POSICAO_DOADOR, coluna_G] == 1.0)
tem_T = np.all(X_treino_oh[:, POSICAO_DOADOR + 1, coluna_T] == 1.0)
print("Todo exemplo tem GT na posição 100:", bool(tem_G and tem_T))

Os três conjuntos já vêm separados no arquivo. A divisão foi feita por
cromossomo, e não por sorteio. O porquê é assunto da seção 2.

In [ ]:
for nome in ("train", "val", "test"):
    rotulos = dados[f"y_{nome}"]
    positivos = int(rotulos.sum())
    print(f"{nome:>6}: {br(len(rotulos)):>9} exemplos, "
          f"{br(positivos):>6} positivos "
          f"({positivos / len(rotulos) * 100:5.2f}%)")

Duas leituras da última coluna:

- validação e teste têm 1,96% de positivos, que é a proporção real do genoma:
  cerca de 1 sítio a cada 51 `GT` dentro de genes;
- treino tem 9,09% porque ali os negativos foram subamostrados de propósito,
  para o treino ficar mais barato.

Dá para baratear o treino. Não dá para baratear a avaliação sem mentir para si
mesmo.

Com 1,96% de positivos, um classificador que respondesse "não é sítio" para
tudo acertaria **98%** das vezes sem olhar para o DNA uma única vez. É por isso
que a seção 2 trata de como medir, antes de treinar qualquer modelo.